# Week 11: Time Series Analysis — Applied Statistics with Python (2026)

Many real-world datasets are ordered by time: stock prices, temperature records, monthly sales, website traffic. **Time series analysis** exploits this temporal structure to uncover patterns and make forecasts. This week we learn to decompose time series into **trend**, **seasonality**, and **noise**, test for **stationarity**, and build forecasting models from simple moving averages to **ARIMA**.

| # | Topic | Key Concepts |
|---|-------|--------------|
| 1 | Time Series Basics | Components, pandas datetime, resampling |
| 2 | Visualization | Line plots, seasonal plots, autocorrelation |
| 3 | Decomposition | Additive vs multiplicative, `seasonal_decompose` |
| 4 | Stationarity | What it is, why it matters, ADF test |
| 5 | Differencing & Transformations | Making series stationary |
| 6 | Autocorrelation (ACF & PACF) | Identifying lag dependencies |
| 7 | Moving Average & Exponential Smoothing | Simple forecasting methods |
| 8 | ARIMA Models | AR, MA, ARIMA, parameter selection |
| 9 | Model Evaluation | Train/test split, RMSE, MAPE |
| 10 | Practical Example | End-to-end forecasting pipeline |
| 11 | Summary | Key functions and concepts |
| 12 | Homework | Practice exercises |

### Import all necessary libraries for this week's analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Time series specific
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
np.random.seed(42)

print('All libraries loaded successfully!')

---
## 1. Time Series Basics

A **time series** is a sequence of data points indexed in time order. Unlike cross-sectional data, the **order matters** — each observation depends on previous observations.

### Components of a Time Series

| Component | Description | Example |
|-----------|-------------|---------|
| **Trend (T)** | Long-term increase or decrease | Population growth |
| **Seasonality (S)** | Regular periodic patterns | Holiday retail sales |
| **Cyclical (C)** | Irregular long-term oscillations | Business cycles |
| **Residual (R)** | Random noise after removing above | Unpredictable fluctuations |

Two decomposition models:
- **Additive**: Y = T + S + R (seasonal amplitude is constant)
- **Multiplicative**: Y = T × S × R (seasonal amplitude grows with trend)

### 1.1 Creating Time Series in Pandas

Pandas has excellent datetime support. The key is using a `DatetimeIndex`.

Demonstrate creating time series with pandas: date ranges, frequency codes, and DatetimeIndex.

In [ ]:
# Creating date ranges
daily = pd.date_range(start='2020-01-01', periods=365, freq='D')   # Daily
monthly = pd.date_range(start='2020-01', periods=36, freq='MS')     # Month Start
quarterly = pd.date_range(start='2020-01', periods=12, freq='QS')   # Quarter Start

print('Common frequency codes:')
freq_table = pd.DataFrame({
    'Code': ['D', 'W', 'MS', 'ME', 'QS', 'YS', 'h', 'min', 'B'],
    'Meaning': ['Calendar day', 'Weekly', 'Month start', 'Month end', 
                'Quarter start', 'Year start', 'Hourly', 'Minutely', 'Business day']
})
print(freq_table.to_string(index=False))

# Create a time series with DatetimeIndex
np.random.seed(42)
ts = pd.Series(
    data=np.cumsum(np.random.randn(365)) + 100,  # Random walk
    index=daily,
    name='value'
)

print(f'\nTime series shape: {ts.shape}')
print(f'Index type: {type(ts.index).__name__}')
print(f'Frequency: {ts.index.freq}')
print(f'Date range: {ts.index[0].date()} to {ts.index[-1].date()}')
print(f'\nFirst 5 values:')
print(ts.head())

### 1.2 Resampling and Aggregation

Time series can be **resampled** to different frequencies: daily → weekly, hourly → daily, etc.

Show how to resample time series to different frequencies using pandas `.resample()` method.

In [ ]:
# Resample: daily → weekly / monthly
ts_weekly = ts.resample('W').mean()     # Weekly average
ts_monthly = ts.resample('MS').mean()    # Monthly average

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ts, alpha=0.3, color='gray', label='Daily')
ax.plot(ts_weekly, color='steelblue', linewidth=1.5, label='Weekly avg')
ax.plot(ts_monthly, color='red', linewidth=2.5, marker='o', markersize=5, label='Monthly avg')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Value', fontsize=12)
ax.set_title('Time Series at Different Frequencies', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Other useful aggregations
print('Monthly statistics:')
print(ts.resample('MS').agg(['mean', 'std', 'min', 'max']).head())

### 1.3 Generating a Realistic Time Series

Let's create a synthetic monthly sales dataset with all four components: trend, seasonality, cyclical pattern, and noise.

Build a synthetic time series by combining trend, seasonality, and noise — we'll use this dataset throughout the chapter.

In [ ]:
# Create 6 years of monthly sales data
np.random.seed(42)
dates = pd.date_range('2018-01', periods=72, freq='MS')  # 6 years × 12 months
t = np.arange(72)

# Components
trend = 200 + 3 * t                            # Linear upward trend
seasonal = 40 * np.sin(2 * np.pi * t / 12)      # Yearly seasonality (peak in summer)
noise = np.random.normal(0, 15, 72)              # Random noise

sales = trend + seasonal + noise
sales_ts = pd.Series(sales, index=dates, name='sales')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(dates, trend, 'r-', linewidth=2)
axes[0, 0].set_title('Trend Component', fontsize=13)

axes[0, 1].plot(dates, seasonal, 'b-', linewidth=2)
axes[0, 1].set_title('Seasonal Component', fontsize=13)

axes[1, 0].plot(dates, noise, 'gray', linewidth=1)
axes[1, 0].set_title('Noise Component', fontsize=13)

axes[1, 1].plot(dates, sales, 'k-', linewidth=1.5)
axes[1, 1].set_title('Combined: Sales = Trend + Seasonal + Noise', fontsize=13)

for ax in axes.flat:
    ax.set_xlabel('Date', fontsize=10)

plt.suptitle('Anatomy of a Time Series', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 2. Time Series Visualization

Good visualization is crucial for understanding time series patterns. Key plots include:
- **Line plot**: Basic trend and pattern
- **Seasonal plot**: Compare the same period across years
- **Box plot by period**: Distribution within each month/quarter
- **Lag plot**: Check for autocorrelation

Create a 4-panel diagnostic visualization for our sales time series.

In [ ]:
sales_df = sales_ts.to_frame()
sales_df['year'] = sales_df.index.year
sales_df['month'] = sales_df.index.month
sales_df['month_name'] = sales_df.index.strftime('%b')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Standard line plot
ax = axes[0, 0]
ax.plot(sales_ts, color='steelblue', linewidth=1.5)
ax.set_title('Monthly Sales Over Time', fontsize=13)
ax.set_ylabel('Sales ($k)', fontsize=11)

# 2. Seasonal plot: each year as a separate line
ax = axes[0, 1]
colors = plt.cm.viridis(np.linspace(0, 1, sales_df['year'].nunique()))
for i, (year, group) in enumerate(sales_df.groupby('year')):
    ax.plot(group['month'], group['sales'], marker='o', linewidth=2, 
            color=colors[i], label=str(year), markersize=4)
ax.set_title('Seasonal Plot (by Year)', fontsize=13)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Sales ($k)', fontsize=11)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
ax.legend(fontsize=9, ncol=2)

# 3. Box plot by month
ax = axes[1, 0]
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
sns.boxplot(data=sales_df, x='month_name', y='sales', ax=ax, 
            order=month_order, palette='coolwarm')
ax.set_title('Distribution by Month', fontsize=13)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Sales ($k)', fontsize=11)

# 4. Lag plot (y_t vs y_{t-1})
ax = axes[1, 1]
ax.scatter(sales_ts.values[:-1], sales_ts.values[1:], alpha=0.5, color='steelblue', s=30)
ax.set_xlabel('Sales at time t', fontsize=11)
ax.set_ylabel('Sales at time t+1', fontsize=11)
ax.set_title(f'Lag Plot (lag=1)\nr = {np.corrcoef(sales_ts.values[:-1], sales_ts.values[1:])[0,1]:.3f}', fontsize=13)

plt.tight_layout()
plt.show()

---
## 3. Time Series Decomposition

**Decomposition** separates a time series into its components:

- **Additive**: Y(t) = Trend(t) + Seasonal(t) + Residual(t)
- **Multiplicative**: Y(t) = Trend(t) × Seasonal(t) × Residual(t)

**Use additive** when seasonal fluctuations are roughly constant in magnitude.  
**Use multiplicative** when seasonal fluctuations grow proportionally with the trend.

Python: `statsmodels.tsa.seasonal.seasonal_decompose()`

Decompose our sales time series using both additive and multiplicative models and compare the results.

In [ ]:
# Additive decomposition
decomp_add = seasonal_decompose(sales_ts, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)

decomp_add.observed.plot(ax=axes[0], color='steelblue')
axes[0].set_ylabel('Observed', fontsize=11)
axes[0].set_title('Additive Decomposition', fontsize=14, fontweight='bold')

decomp_add.trend.plot(ax=axes[1], color='red')
axes[1].set_ylabel('Trend', fontsize=11)

decomp_add.seasonal.plot(ax=axes[2], color='green')
axes[2].set_ylabel('Seasonal', fontsize=11)

decomp_add.resid.plot(ax=axes[3], color='gray')
axes[3].set_ylabel('Residual', fontsize=11)
axes[3].set_xlabel('Date', fontsize=11)

plt.tight_layout()
plt.show()

# Seasonal factors
seasonal_pattern = decomp_add.seasonal[:12]
print('Monthly seasonal factors (additive):')
for i, val in enumerate(seasonal_pattern):
    month = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
             'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'][i]
    print(f'  {month}: {val:+.1f}')

Create a multiplicative example with growing seasonal amplitude.

In [ ]:
# Multiplicative example: seasonal amplitude grows with trend
np.random.seed(42)
trend_mult = 100 * np.exp(0.02 * t)  # Exponential trend
seasonal_mult = 1 + 0.2 * np.sin(2 * np.pi * t / 12)  # Multiplicative seasonal (±20%)
noise_mult = 1 + np.random.normal(0, 0.05, 72)  # Multiplicative noise
sales_mult = trend_mult * seasonal_mult * noise_mult
sales_mult_ts = pd.Series(sales_mult, index=dates, name='sales')

# Compare additive vs multiplicative decomposition
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Wrong: additive on multiplicative data
decomp_wrong = seasonal_decompose(sales_mult_ts, model='additive', period=12)
decomp_wrong.resid.plot(ax=axes[0], color='coral', linewidth=1)
axes[0].set_title('Additive on Multiplicative Data\n(residuals show pattern = BAD)', fontsize=12, color='red')
axes[0].set_ylabel('Residual', fontsize=11)

# Correct: multiplicative on multiplicative data
decomp_right = seasonal_decompose(sales_mult_ts, model='multiplicative', period=12)
decomp_right.resid.plot(ax=axes[1], color='seagreen', linewidth=1)
axes[1].set_title('Multiplicative on Multiplicative Data\n(residuals are random = GOOD)', fontsize=12, color='green')
axes[1].set_ylabel('Residual', fontsize=11)

plt.tight_layout()
plt.show()

---
## 4. Stationarity

A time series is **stationary** if its statistical properties (mean, variance, autocorrelation) don't change over time.

**Why does stationarity matter?** Most forecasting models (ARIMA, etc.) assume stationarity. Non-stationary series must be transformed first.

| Property | Stationary | Non-stationary |
|----------|-----------|----------------|
| Mean | Constant | Changes over time (trend) |
| Variance | Constant | Changes over time |
| Autocorrelation | Depends only on lag | Changes over time |

### Augmented Dickey-Fuller (ADF) Test
- H₀: The series has a **unit root** (non-stationary)
- H₁: The series is **stationary**
- If p < 0.05 → Reject H₀ → Series IS stationary

Create a reusable function to test stationarity and apply it to different types of time series.

In [ ]:
def test_stationarity(series, title='', window=12):
    """Test stationarity with rolling stats and ADF test."""
    # Rolling statistics
    rolling_mean = series.rolling(window=window).mean()
    rolling_std = series.rolling(window=window).std()
    
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(series, color='steelblue', alpha=0.6, label='Original')
    ax.plot(rolling_mean, color='red', linewidth=2, label=f'Rolling Mean ({window})')
    ax.plot(rolling_std, color='orange', linewidth=2, label=f'Rolling Std ({window})')
    ax.legend(fontsize=10)
    ax.set_title(f'Stationarity Check: {title}', fontsize=13)
    plt.tight_layout()
    plt.show()
    
    # ADF test
    result = adfuller(series.dropna(), autolag='AIC')
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value:       {result[1]:.4f}')
    print(f'Critical values:')
    for key, val in result[4].items():
        print(f'  {key}: {val:.4f}')
    print(f'\nConclusion: {"STATIONARY" if result[1] < 0.05 else "NON-STATIONARY"} (p {"<" if result[1] < 0.05 else ">="} 0.05)')
    return result[1]

# Test our sales data
print('=== Original Sales Series ===')
p1 = test_stationarity(sales_ts, 'Monthly Sales (Original)')

Compare stationary vs non-stationary series visually.

In [ ]:
# Generate examples of stationary and non-stationary series
np.random.seed(42)
n = 200
dates_demo = pd.date_range('2020-01', periods=n, freq='D')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 1. Stationary: white noise
white_noise = pd.Series(np.random.normal(0, 1, n), index=dates_demo)
axes[0, 0].plot(white_noise, color='steelblue', linewidth=0.8)
adf_p = adfuller(white_noise)[1]
axes[0, 0].set_title(f'White Noise (Stationary)\nADF p = {adf_p:.4f}', fontsize=12)

# 2. Non-stationary: random walk
random_walk = pd.Series(np.cumsum(np.random.normal(0, 1, n)), index=dates_demo)
adf_p2 = adfuller(random_walk)[1]
axes[0, 1].plot(random_walk, color='coral', linewidth=1.2)
axes[0, 1].set_title(f'Random Walk (Non-stationary)\nADF p = {adf_p2:.4f}', fontsize=12)

# 3. Stationary: mean-reverting (AR(1) with |φ|<1)
ar1 = [0]
for i in range(1, n):
    ar1.append(0.7 * ar1[-1] + np.random.normal(0, 1))
ar1_series = pd.Series(ar1, index=dates_demo)
adf_p3 = adfuller(ar1_series)[1]
axes[1, 0].plot(ar1_series, color='seagreen', linewidth=0.8)
axes[1, 0].set_title(f'AR(1) φ=0.7 (Stationary)\nADF p = {adf_p3:.4f}', fontsize=12)

# 4. Non-stationary: trend + seasonality
axes[1, 1].plot(sales_ts, color='purple', linewidth=1.2)
adf_p4 = adfuller(sales_ts)[1]
axes[1, 1].set_title(f'Trend + Seasonality (Non-stationary)\nADF p = {adf_p4:.4f}', fontsize=12)

for ax in axes.flat:
    ax.set_xlabel('Date', fontsize=10)

plt.suptitle('Stationary vs Non-Stationary Series', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 5. Differencing & Transformations

To make a non-stationary series stationary:

| Technique | Removes | Formula |
|-----------|---------|--------|
| **First differencing** | Linear trend | y'(t) = y(t) - y(t-1) |
| **Second differencing** | Quadratic trend | y''(t) = y'(t) - y'(t-1) |
| **Seasonal differencing** | Seasonality | y*(t) = y(t) - y(t-m) |
| **Log transform** | Increasing variance | y_log(t) = log(y(t)) |

The number of regular differences needed is the **d** in ARIMA(p,d,q).  
The number of seasonal differences is the **D** in SARIMA.

Apply first differencing and seasonal differencing to our sales data and check stationarity after each transformation.

In [ ]:
# First differencing: removes trend
sales_diff1 = sales_ts.diff().dropna()

# Seasonal differencing (lag=12): removes seasonality
sales_seasonal_diff = sales_ts.diff(12).dropna()

# Both: first difference of seasonal difference
sales_both = sales_ts.diff(12).diff().dropna()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

series_list = [
    (sales_ts, 'Original'),
    (sales_diff1, 'First Difference (d=1)'),
    (sales_seasonal_diff, 'Seasonal Difference (D=1, m=12)'),
    (sales_both, 'Both (d=1, D=1)')
]

for ax, (s, title) in zip(axes.flat, series_list):
    ax.plot(s, color='steelblue', linewidth=1)
    adf_p = adfuller(s.dropna())[1]
    ax.set_title(f'{title}\nADF p = {adf_p:.4f} ({"Stationary" if adf_p < 0.05 else "Non-stationary"})', 
                 fontsize=12, color='green' if adf_p < 0.05 else 'red')
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Making a Series Stationary Through Differencing', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Autocorrelation: ACF & PACF

**Autocorrelation** measures how a series correlates with lagged versions of itself.

| Plot | Full Name | Measures | Used For |
|------|-----------|---------|----------|
| **ACF** | Autocorrelation Function | Correlation at lag k (includes indirect) | Identifies MA order (q) |
| **PACF** | Partial Autocorrelation Function | Direct correlation at lag k only | Identifies AR order (p) |

### Reading ACF/PACF for ARIMA

| Pattern | ACF | PACF | Model |
|---------|-----|------|-------|
| Cuts off at lag q | Sharp cutoff | Gradual decay | MA(q) |
| Gradual decay | Gradual decay | Sharp cutoff at lag p | AR(p) |
| Both decay | Gradual | Gradual | ARMA(p,q) |

Plot ACF and PACF for both the original and differenced sales series.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# ACF and PACF of original series
plot_acf(sales_ts, lags=30, ax=axes[0, 0], title='ACF — Original Sales')
plot_pacf(sales_ts, lags=30, ax=axes[0, 1], title='PACF — Original Sales', method='ywm')

# ACF and PACF of differenced series
plot_acf(sales_both.dropna(), lags=30, ax=axes[1, 0], title='ACF — After Differencing (d=1, D=1)')
plot_pacf(sales_both.dropna(), lags=30, ax=axes[1, 1], title='PACF — After Differencing', method='ywm')

plt.tight_layout()
plt.show()

print('Observations:')
print('• Original: ACF decays slowly (non-stationary) with seasonal spikes at 12, 24')
print('• Differenced: ACF/PACF mostly within confidence bands (stationary)')
print('• Significant spikes at lag 1 suggest AR(1) or MA(1) component')

---
## 7. Moving Average & Exponential Smoothing

These are simple but effective forecasting methods.

### 7.1 Simple Moving Average (SMA)
Average of the last k observations: great for smoothing, but lags behind trends.

### 7.2 Exponential Smoothing
Weighted average giving **more weight to recent** observations:

| Method | Handles | Parameters |
|--------|---------|------------|
| **Simple (SES)** | Level only | α (smoothing) |
| **Double (Holt)** | Level + trend | α, β |
| **Triple (Holt-Winters)** | Level + trend + seasonality | α, β, γ |

Apply simple moving average with different window sizes and compare the smoothing effect.

In [ ]:
# Simple Moving Average with different windows
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(sales_ts, color='gray', alpha=0.5, linewidth=1, label='Original')
for window, color in [(3, 'steelblue'), (6, 'coral'), (12, 'seagreen')]:
    sma = sales_ts.rolling(window=window).mean()
    ax.plot(sma, color=color, linewidth=2, label=f'SMA({window})')

ax.set_title('Simple Moving Average: Smoothing Effect', fontsize=14)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Sales', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('Trade-off: Larger window → smoother but more lag')

### 7.3 Holt-Winters Exponential Smoothing

Holt-Winters handles trend AND seasonality — the most practical exponential smoothing method.

Fit Holt-Winters models (both additive and multiplicative) and generate forecasts.

In [ ]:
# Split data: train on first 60 months, test on last 12
train = sales_ts[:60]
test = sales_ts[60:]

# Holt-Winters: additive seasonality
hw_add = ExponentialSmoothing(train, seasonal_periods=12, trend='add', 
                               seasonal='add').fit(optimized=True)
forecast_add = hw_add.forecast(12)

# Simple Exponential Smoothing (no trend/seasonality — for comparison)
ses = SimpleExpSmoothing(train).fit(optimized=True)
forecast_ses = ses.forecast(12)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train, color='steelblue', linewidth=1.5, label='Train')
ax.plot(test, color='gray', linewidth=2, label='Actual (test)')
ax.plot(forecast_add, color='red', linewidth=2, linestyle='--', marker='o', 
        markersize=4, label='Holt-Winters')
ax.plot(forecast_ses, color='orange', linewidth=2, linestyle=':', label='Simple Exp Smoothing')
ax.axvline(train.index[-1], color='black', linestyle=':', alpha=0.5)
ax.set_title('Forecasting: Holt-Winters vs Simple Exponential Smoothing', fontsize=14)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Sales', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Accuracy metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

def forecast_metrics(actual, predicted, name=''):
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    print(f'{name:>25}: RMSE={rmse:.2f}, MAE={mae:.2f}, MAPE={mape:.1f}%')

print('\nForecast Accuracy (12-month test set):')
forecast_metrics(test, forecast_add, 'Holt-Winters (additive)')
forecast_metrics(test, forecast_ses, 'Simple Exp Smoothing')

---
## 8. ARIMA Models

**ARIMA** (AutoRegressive Integrated Moving Average) is the most widely used time series model.

### Components: ARIMA(p, d, q)

| Parameter | Name | Meaning |
|-----------|------|---------|
| **p** | AR order | Number of lagged values (autoregressive terms) |
| **d** | Differencing | Number of times the series is differenced |
| **q** | MA order | Number of lagged forecast errors (moving average terms) |

### Model Equations
- **AR(p)**: y(t) = c + φ₁y(t-1) + φ₂y(t-2) + ... + φₚy(t-p) + ε(t)
- **MA(q)**: y(t) = c + ε(t) + θ₁ε(t-1) + θ₂ε(t-2) + ... + θᵧε(t-q)
- **ARIMA(p,d,q)**: Apply AR(p) and MA(q) to the d-times differenced series

### 8.1 Understanding AR and MA Processes

Let's simulate pure AR and MA processes to see their ACF/PACF signatures.

Simulate AR(1), AR(2), MA(1), and MA(2) processes and examine their ACF/PACF patterns.

In [ ]:
np.random.seed(42)
n = 300

# AR(1): φ = 0.8
ar1_sim = [0]
for i in range(1, n):
    ar1_sim.append(0.8 * ar1_sim[-1] + np.random.normal(0, 1))

# MA(1): θ = 0.8
errors = np.random.normal(0, 1, n)
ma1_sim = [errors[0]]
for i in range(1, n):
    ma1_sim.append(errors[i] + 0.8 * errors[i-1])

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# AR(1)
axes[0, 0].plot(ar1_sim, color='steelblue', linewidth=0.8)
axes[0, 0].set_title('AR(1): φ = 0.8', fontsize=12)
plot_acf(ar1_sim, lags=20, ax=axes[0, 1], title='ACF — AR(1): Decays gradually')
plot_pacf(ar1_sim, lags=20, ax=axes[0, 2], title='PACF — AR(1): Cuts off at lag 1', method='ywm')

# MA(1)
axes[1, 0].plot(ma1_sim, color='coral', linewidth=0.8)
axes[1, 0].set_title('MA(1): θ = 0.8', fontsize=12)
plot_acf(ma1_sim, lags=20, ax=axes[1, 1], title='ACF — MA(1): Cuts off at lag 1')
plot_pacf(ma1_sim, lags=20, ax=axes[1, 2], title='PACF — MA(1): Decays gradually', method='ywm')

plt.tight_layout()
plt.show()

print('Key pattern:')
print('• AR(p): ACF decays, PACF cuts off → tells you p')
print('• MA(q): ACF cuts off, PACF decays → tells you q')

### 8.2 Fitting ARIMA Models

Let's fit ARIMA models to our sales data.

Fit ARIMA with different (p,d,q) orders and compare their AIC values to find the best specification.

In [ ]:
# Grid search for best ARIMA order
best_aic = np.inf
best_order = None
results_list = []

for p in range(0, 4):
    for d in range(0, 3):
        for q in range(0, 4):
            try:
                model = ARIMA(train, order=(p, d, q)).fit()
                results_list.append({
                    'Order': f'({p},{d},{q})',
                    'AIC': model.aic,
                    'BIC': model.bic
                })
                if model.aic < best_aic:
                    best_aic = model.aic
                    best_order = (p, d, q)
            except:
                continue

results_df = pd.DataFrame(results_list).sort_values('AIC').head(10)
print('=== Top 10 ARIMA Models by AIC ===')
print(results_df.to_string(index=False))
print(f'\nBest order: ARIMA{best_order} with AIC = {best_aic:.2f}')

Fit the best ARIMA model, examine its summary, and generate forecasts.

In [ ]:
# Fit best ARIMA model
best_model = ARIMA(train, order=best_order).fit()
print(best_model.summary())

Generate ARIMA forecasts with confidence intervals and compare to actual values.

In [ ]:
# Forecast
forecast_result = best_model.get_forecast(steps=12)
forecast_arima = forecast_result.predicted_mean
forecast_ci = forecast_result.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train, color='steelblue', linewidth=1.5, label='Train')
ax.plot(test, color='gray', linewidth=2, label='Actual')
ax.plot(forecast_arima, color='red', linewidth=2, linestyle='--', label=f'ARIMA{best_order}')
ax.fill_between(forecast_ci.index, forecast_ci.iloc[:, 0], forecast_ci.iloc[:, 1],
                color='red', alpha=0.15, label='95% CI')
ax.axvline(train.index[-1], color='black', linestyle=':', alpha=0.5)
ax.set_title(f'ARIMA{best_order} Forecast', fontsize=14)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Sales', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

forecast_metrics(test, forecast_arima, f'ARIMA{best_order}')

### 8.3 Checking Residuals

A good model should have residuals that look like white noise: no autocorrelation, mean zero, normally distributed.

Diagnose the ARIMA residuals with ACF, histogram, and the Ljung-Box test for autocorrelation.

In [ ]:
residuals = best_model.resid

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Residual time plot
axes[0, 0].plot(residuals, color='steelblue', linewidth=0.8)
axes[0, 0].axhline(0, color='red', linestyle='--')
axes[0, 0].set_title('Residuals Over Time', fontsize=12)

# Histogram
axes[0, 1].hist(residuals, bins=20, density=True, color='steelblue', alpha=0.7, edgecolor='white')
x_r = np.linspace(residuals.min(), residuals.max(), 100)
axes[0, 1].plot(x_r, stats.norm.pdf(x_r, residuals.mean(), residuals.std()), 'r-', linewidth=2)
axes[0, 1].set_title('Residual Distribution', fontsize=12)

# ACF of residuals
plot_acf(residuals, lags=20, ax=axes[1, 0], title='ACF of Residuals')

# Q-Q plot
stats.probplot(residuals, plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot of Residuals', fontsize=12)

plt.suptitle('ARIMA Residual Diagnostics', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Ljung-Box test: H₀ = no autocorrelation in residuals
from statsmodels.stats.diagnostic import acorr_ljungbox
lb_result = acorr_ljungbox(residuals, lags=10, return_df=True)
print('Ljung-Box Test (H₀: no autocorrelation):')
print(lb_result.round(4))
print(f'\nAll p > 0.05? {(lb_result["lb_pvalue"] > 0.05).all()} → Residuals are white noise.')

---
## 9. Model Evaluation

### Time Series Train/Test Split

**Never use random splits** for time series! Always use **chronological splits** — train on past, test on future.

### Forecast Accuracy Metrics

| Metric | Formula | Interpretation |
|--------|---------|---------------|
| **RMSE** | √(mean(e²)) | Penalizes large errors more |
| **MAE** | mean(\|e\|) | Average absolute error |
| **MAPE** | mean(\|e/y\|)×100 | Percentage error (scale-free) |
| **R²** | 1 - SS_res/SS_total | Proportion of variance explained |

Compare all forecasting methods we've learned on the same train/test split.

In [ ]:
# Compare all methods
print('=== Forecast Method Comparison (12-month test set) ===')
print(f'{"Method":>30} {"RMSE":>8} {"MAE":>8} {"MAPE%":>8}')
print('-' * 58)

methods = {}

# 1. Naive (last value)
naive_fc = pd.Series([train.iloc[-1]] * 12, index=test.index)
methods['Naive (last value)'] = naive_fc

# 2. Seasonal naive (same month last year)
seasonal_naive = train.iloc[-12:].values
seasonal_naive_fc = pd.Series(seasonal_naive, index=test.index)
methods['Seasonal Naive'] = seasonal_naive_fc

# 3. SES
methods['Simple Exp Smoothing'] = forecast_ses

# 4. Holt-Winters
methods['Holt-Winters (add)'] = forecast_add

# 5. ARIMA
methods[f'ARIMA{best_order}'] = forecast_arima

for name, fc in methods.items():
    rmse = np.sqrt(mean_squared_error(test, fc))
    mae = mean_absolute_error(test, fc)
    mape = np.mean(np.abs((test.values - fc.values) / test.values)) * 100
    print(f'{name:>30} {rmse:>8.2f} {mae:>8.2f} {mape:>7.1f}%')

Visualize all forecast methods against actual values.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(train[-24:], color='steelblue', linewidth=1.5, label='Train (last 2yr)')
ax.plot(test, color='black', linewidth=2.5, label='Actual')

colors = ['gray', 'orange', 'purple', 'red', 'green']
styles = [':', '-.', '--', '-', '-']
for (name, fc), color, style in zip(methods.items(), colors, styles):
    ax.plot(fc, color=color, linewidth=1.8, linestyle=style, label=name)

ax.axvline(train.index[-1], color='black', linestyle=':', alpha=0.5)
ax.set_title('Forecast Comparison: All Methods', fontsize=14)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Sales', fontsize=12)
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

---
## 10. Practical Example: Airline Passenger Forecasting

Let's apply the full time series analysis pipeline to a classic dataset: monthly airline passenger counts.

### Step 1: Load and Explore the Data

Generate a realistic airline passenger dataset and perform initial exploration.

In [ ]:
# Simulate airline passenger data (similar to classic Box-Jenkins dataset)
np.random.seed(42)
dates_air = pd.date_range('2010-01', periods=120, freq='MS')  # 10 years
t_air = np.arange(120)

# Multiplicative: growing trend with growing seasonal amplitude
trend_air = 200 + 2.5 * t_air + 0.02 * t_air**2   # Accelerating trend
seasonal_air = 1 + 0.15 * np.sin(2 * np.pi * t_air / 12 - np.pi/6)  # Summer peak
noise_air = 1 + np.random.normal(0, 0.03, 120)

passengers = trend_air * seasonal_air * noise_air
passengers_ts = pd.Series(passengers.round(0), index=dates_air, name='passengers')

print(f'Date range: {passengers_ts.index[0].strftime("%Y-%m")} to {passengers_ts.index[-1].strftime("%Y-%m")}')
print(f'Observations: {len(passengers_ts)}')
print(f'\nDescriptive Statistics:')
print(passengers_ts.describe().round(1))

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(passengers_ts, color='steelblue', linewidth=1.5)
ax.set_title('Monthly Airline Passengers (2010-2019)', fontsize=14)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Passengers (thousands)', fontsize=12)
plt.tight_layout()
plt.show()

### Step 2: Decomposition and Stationarity

Decompose the series, check stationarity, and apply transformations to achieve stationarity.

In [ ]:
# Multiplicative decomposition (seasonal amplitude grows with trend)
decomp_air = seasonal_decompose(passengers_ts, model='multiplicative', period=12)

fig = decomp_air.plot()
fig.set_size_inches(13, 10)
fig.suptitle('Multiplicative Decomposition', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Log transform to stabilize variance, then difference
log_passengers = np.log(passengers_ts)
log_diff = log_passengers.diff(12).diff().dropna()  # Seasonal + first diff

print('Stationarity tests:')
print(f'  Original:       ADF p = {adfuller(passengers_ts)[1]:.4f}')
print(f'  Log transform:  ADF p = {adfuller(log_passengers)[1]:.4f}')
print(f'  Log + diff:     ADF p = {adfuller(log_diff)[1]:.4f} → Stationary!')

### Step 3: Model Building and Forecasting

Split data, fit Holt-Winters and ARIMA models, and compare forecasts.

In [ ]:
# Train/test split: last 12 months for testing
train_air = passengers_ts[:108]
test_air = passengers_ts[108:]

# Method 1: Holt-Winters (multiplicative)
hw_model = ExponentialSmoothing(train_air, seasonal_periods=12, 
                                 trend='add', seasonal='mul').fit(optimized=True)
hw_forecast = hw_model.forecast(12)

# Method 2: ARIMA on log-transformed data
log_train = np.log(train_air)

# Quick grid search on log data
best_aic_air = np.inf
best_order_air = None
for p in range(0, 3):
    for d in range(1, 3):
        for q in range(0, 3):
            try:
                m = ARIMA(log_train, order=(p, d, q)).fit()
                if m.aic < best_aic_air:
                    best_aic_air = m.aic
                    best_order_air = (p, d, q)
            except:
                continue

arima_model = ARIMA(log_train, order=best_order_air).fit()
arima_fc_log = arima_model.forecast(12)
arima_fc = np.exp(arima_fc_log)  # Transform back

# Method 3: Seasonal naive
sn_fc = pd.Series(train_air.iloc[-12:].values, index=test_air.index)

# Compare
print(f'Best ARIMA order: {best_order_air}')
print(f'\n{"Method":>25} {"RMSE":>8} {"MAE":>8} {"MAPE%":>8}')
print('-' * 52)
for name, fc in [('Seasonal Naive', sn_fc), 
                  ('Holt-Winters (mul)', hw_forecast),
                  (f'ARIMA{best_order_air}', arima_fc)]:
    rmse = np.sqrt(mean_squared_error(test_air, fc))
    mae = mean_absolute_error(test_air, fc)
    mape = np.mean(np.abs((test_air.values - fc.values) / test_air.values)) * 100
    print(f'{name:>25} {rmse:>8.1f} {mae:>8.1f} {mape:>7.1f}%')

### Step 4: Visualize Final Results

Create a comprehensive visualization comparing all forecasting methods.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Top: full view with forecasts
ax = axes[0]
ax.plot(train_air, color='steelblue', linewidth=1.2, label='Train')
ax.plot(test_air, color='black', linewidth=2.5, label='Actual')
ax.plot(hw_forecast, color='red', linewidth=2, linestyle='--', label='Holt-Winters')
ax.plot(arima_fc, color='green', linewidth=2, linestyle='-.', label=f'ARIMA{best_order_air}')
ax.plot(sn_fc, color='orange', linewidth=1.5, linestyle=':', label='Seasonal Naive')
ax.axvline(train_air.index[-1], color='black', linestyle=':', alpha=0.3)
ax.set_title('Airline Passengers: Forecast Comparison', fontsize=14)
ax.set_ylabel('Passengers (k)', fontsize=12)
ax.legend(fontsize=10)

# Bottom: zoomed on test period
ax = axes[1]
ax.plot(test_air, color='black', linewidth=2.5, marker='o', markersize=6, label='Actual')
ax.plot(hw_forecast, color='red', linewidth=2, marker='s', markersize=5, label='Holt-Winters')
ax.plot(arima_fc, color='green', linewidth=2, marker='^', markersize=5, label=f'ARIMA{best_order_air}')
ax.set_title('Test Period (Zoomed)', fontsize=14)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Passengers (k)', fontsize=12)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## 11. Summary

| Section | Key Concepts | Python Functions |
|---------|-------------|------------------|
| 1. Basics | Components (T, S, R), DatetimeIndex, resampling | `pd.date_range()`, `.resample()` |
| 2. Visualization | Line, seasonal, box, lag plots | `matplotlib`, `seaborn` |
| 3. Decomposition | Additive vs multiplicative | `seasonal_decompose()` |
| 4. Stationarity | Constant mean/variance, ADF test | `adfuller()` |
| 5. Differencing | Remove trend/seasonality | `.diff()`, `.diff(12)` |
| 6. ACF & PACF | Lag dependencies, model identification | `plot_acf()`, `plot_pacf()` |
| 7. Exp Smoothing | SES, Holt, Holt-Winters | `ExponentialSmoothing()` |
| 8. ARIMA | AR + I + MA, parameter selection | `ARIMA(order=(p,d,q))` |
| 9. Evaluation | RMSE, MAE, MAPE, chronological split | `mean_squared_error()` |
| 10. Practical | End-to-end forecasting pipeline | All above combined |

### Key Takeaways

1. **Always visualize** your time series first — look for trend, seasonality, outliers
2. **Decomposition** helps you understand the components driving the series
3. **Stationarity is essential** for ARIMA — use differencing and the ADF test
4. **ACF/PACF patterns** guide the choice of p, d, q for ARIMA
5. **Holt-Winters** is often the best simple method for seasonal data
6. **Never use random train/test splits** for time series — always split chronologically
7. **Compare multiple methods** — no single method is always best

---
## 12. Homework

### Task 1: Time Series Exploration
Generate 5 years of daily temperature data:
```python
np.random.seed(42)
days = pd.date_range('2019-01-01', periods=365*5, freq='D')
t = np.arange(len(days))
temp = 15 + 10*np.sin(2*np.pi*t/365) + 0.005*t + np.random.normal(0, 3, len(days))
temp_ts = pd.Series(temp, index=days, name='temperature')
```
1. Plot the full time series
2. Resample to weekly and monthly averages and plot all three on the same axes
3. Create a seasonal plot (overlay each year)
4. Decompose using additive model (period=365)
5. Is the series stationary? Test with ADF on the original and differenced versions

### Task 2: Exponential Smoothing
Using the monthly temperature (resample from Task 1):
1. Split into train (first 48 months) and test (last 12 months)
2. Fit Simple Exponential Smoothing, Holt (trend only), and Holt-Winters (trend + season)
3. Forecast 12 months with each method
4. Compare RMSE, MAE, and MAPE
5. Visualize all forecasts vs actual on the same plot

### Task 3: ARIMA Modeling
Create a synthetic monthly dataset:
```python
np.random.seed(42)
n = 96  # 8 years
dates = pd.date_range('2016-01', periods=n, freq='MS')
revenue = 500 + 5*np.arange(n) + 30*np.sin(2*np.pi*np.arange(n)/12) + np.random.normal(0, 20, n)
revenue_ts = pd.Series(revenue, index=dates)
```
1. Check stationarity (ADF test)
2. Apply differencing until stationary, noting the d value
3. Plot ACF and PACF of the stationary series to identify p and q
4. Fit ARIMA models with 3-4 candidate orders, compare AIC
5. Use the best model to forecast the next 12 months with 95% confidence intervals
6. Check residual diagnostics (ACF, Q-Q plot, Ljung-Box)

### Task 4: Complete Forecasting Project
Generate monthly e-commerce sales with multiplicative seasonality:
```python
np.random.seed(42)
dates = pd.date_range('2017-01', periods=84, freq='MS')  # 7 years
t = np.arange(84)
trend = 100 * np.exp(0.015 * t)
seasonal = 1 + 0.3*np.sin(2*np.pi*t/12) + 0.1*np.sin(4*np.pi*t/12)
ecom = trend * seasonal * (1 + np.random.normal(0, 0.05, 84))
ecom_ts = pd.Series(ecom, index=dates)
```
1. Perform complete EDA (line plot, seasonal plot, box plot by month)
2. Choose additive or multiplicative decomposition (justify your choice)
3. Make the series stationary (log + differencing)
4. Build at least 3 models: Holt-Winters, ARIMA, and seasonal naive
5. Compare all models on 12-month test set
6. Create a final dashboard showing: the data, decomposition, best forecast with CI

---
### Next Week Preview

**Week 12: Bayesian Statistics** — We shift from the frequentist paradigm to the **Bayesian** approach, where probability represents our **degree of belief**. We'll learn about priors, likelihoods, and posteriors, compare Bayesian vs frequentist thinking, and use **PyMC** to perform Bayesian inference on real problems.

---
*Applied Statistics with Python (2026) | Week 11 | Time Series Analysis*